# SQL - Value Errors

Introduce declared-range violations and/or incorrect numeric scaling directly in a disposable PostgreSQL practice table.

**Instructions:** Run `prepare_data_sql_update.ipynb` or the stable SQL preparation script to restore the clean `practice_N.weather_hourly` table. Adjust the schema and configuration. The notebook changes **only the practice table**, never canonical data. Requires PostgreSQL and `psycopg`.

In [ ]:
from random import Random
import re
import pandas as pd
import psycopg
from psycopg import sql

# Configuration
DATABASE_NAME = "analytical_workflow_fluency"
PRACTICE_SCHEMA = "practice_014"  # Match prepare_data.py
PRACTICE_TABLE = "weather_hourly"
TIME_COLUMN = "time"
KEY_COLUMN = "time"  # Unique identifier; not necessarily the time column in business data.
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive local calendar date.
SEED = 814205

# Value Errors
ERROR_TARGET_COLUMN = "relative_humidity_2m"
ERROR_KIND = "out_of_bounds"  # "out_of_bounds", "scaled", "mixed", "random"
ERROR_PLACEMENT = "mixed"  # "scattered", "grouped", "mixed", "random"
ERROR_RATE = (0.01, 0.03)  # Share of current eligible nonmissing/valid values changed.
ERROR_GROUPS = 3
ERROR_GROUPED_SHARE = 0.70
ERROR_BOUNDS_SHARE = 0.50
VALID_MIN = 0
VALID_MAX = 100
OUTSIDE_OFFSET = 12
SCALE_FACTOR = 10

In [ ]:
# Shared SQL helpers: copy these only once in a combined dirty_data.py.
def sql_date_bounds(timezone, start_date, end_date):
    start = pd.Timestamp(start_date, tz=timezone).to_pydatetime()
    end = pd.Timestamp(end_date, tz=timezone).to_pydatetime()
    if start >= end:
        raise ValueError("Start date must precede end date.")
    return start, end


def check_practice_table(conn, schema):
    if not re.fullmatch(r"practice_[0-9]+", schema):
        raise ValueError("Only practice_N schemas can be deliberately corrupted.")
    if conn.autocommit:
        raise ValueError("Use a transaction (autocommit must be off).")

def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_value_errors_sql(
    conn, rng, *, schema, table, time_column, key_column, timezone,
    start_date, end_date, target_column, kind, placement, error_rate,
    n_groups, grouped_share, bounds_share, valid_min, valid_max,
    outside_offset, scale_factor,
):
    check_practice_table(conn, schema)
    if target_column in (time_column, key_column):
        raise ValueError("Cannot corrupt row identity or time.")
    if kind not in ("out_of_bounds", "scaled", "mixed", "random"):
        raise ValueError("Unrecognized value-error kind.")
    if valid_min >= valid_max or outside_offset <= 0 or scale_factor == 1:
        raise ValueError("Invalid bounds, outside offset, or scale factor.")
    if not 0 < bounds_share < 1:
        raise ValueError("Bounds share must be between 0 and 1.")

    start, end = sql_date_bounds(timezone, start_date, end_date)
    source = sql.Identifier(schema, table)
    key_id = sql.Identifier(key_column)
    time_id = sql.Identifier(time_column)
    target_id = sql.Identifier(target_column)
    metadata = conn.execute("""
        SELECT data_type FROM information_schema.columns
        WHERE table_schema=%s AND table_name=%s AND column_name=%s
    """, (schema, table, target_column)).fetchone()
    numeric = {"smallint", "integer", "bigint", "numeric", "real", "double precision", "decimal"}
    if not metadata or metadata[0] not in numeric:
        raise ValueError("Target must be an existing numeric column.")
    if metadata[0] in {"smallint", "integer", "bigint"}:
        if float(outside_offset) % 1 or float(scale_factor) % 1:
            raise ValueError("Use whole-number offsets and factors for integer columns.")

    rows = conn.execute(
        sql.SQL("""
            SELECT {key}, {target} FROM {source}
            WHERE {time} >= %s AND {time} < %s
            ORDER BY {time}, {key}
        """).format(key=key_id, target=target_id, source=source, time=time_id),
        (start, end),
    ).fetchall()
    keys = [k for k, _ in rows]
    if not keys or any(k is None for k in keys) or len(set(keys)) != len(keys):
        raise ValueError("Expected non-NULL unique keys before numeric corruption.")
    # Previous missingness may have set this field to NULL. Do not rewrite those rows.
    # Existing non-NULL violations still require an independently clean starting state.
    if any(value is not None and not valid_min <= value <= valid_max for _, value in rows):
        raise ValueError("Target has out-of-contract non-NULL values in scope.")
    chosen_kind = rng.choice(("out_of_bounds", "scaled", "mixed")) if kind == "random" else kind
    # Scaling zero would introduce no change, so those rows are ineligible for scaled/mixed cases.
    eligible = [(k, v) for k, v in rows if v is not None and
                (chosen_kind == "out_of_bounds" or v != 0)]
    positions, chosen_placement = select_positions(
        len(eligible), rng, method=placement, rate=error_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    selected = [eligible[i][0] for i in positions]
    if chosen_kind == "mixed" and len(selected) < 2:
        raise ValueError("Mixed value errors need at least two affected observations.")
    bounds_count = (len(selected) if chosen_kind == "out_of_bounds" else
                    max(1, min(len(selected) - 1, round(len(selected) * bounds_share)))
                    if chosen_kind == "mixed" else 0)
    bounds_keys = rng.sample(selected, k=bounds_count)
    scaled_keys = [k for k in selected if k not in set(bounds_keys)]
    upper_keys = rng.sample(bounds_keys, k=len(bounds_keys) // 2)
    lower_keys = [k for k in bounds_keys if k not in set(upper_keys)]

    changes = []
    for group, new_value in ((upper_keys, valid_max + outside_offset),
                             (lower_keys, valid_min - outside_offset)):
        if group:
            changes.extend(conn.execute(
                sql.SQL("""
                    UPDATE {source} SET {target} = %s
                    WHERE {key} = ANY(%s) AND {time} >= %s AND {time} < %s
                    RETURNING {key}
                """).format(source=source, target=target_id, key=key_id, time=time_id),
                (new_value, group, start, end),
            ).fetchall())
    if scaled_keys:
        changes.extend(conn.execute(
            sql.SQL("""
                UPDATE {source} SET {target} = {target} * %s
                WHERE {key} = ANY(%s) AND {time} >= %s AND {time} < %s
                RETURNING {key}
            """).format(source=source, target=target_id, key=key_id, time=time_id),
            (scale_factor, scaled_keys, start, end),
        ).fetchall())
    if len(changes) != len(selected) or {row[0] for row in changes} != set(selected):
        raise ValueError("Altered keys do not match selection; roll back transaction.")
    return len(changes), chosen_kind, chosen_placement


In [ ]:
# A value can be syntactically valid but violate a business rule.
example = pd.DataFrame({"discount_pct": [5, 15, 30, 50]})
example.loc[[1, 3], "discount_pct"] = [112, -12]
assert (~example["discount_pct"].between(0, 100)).sum() == 2
example

# Seeded selection behavior, independently of the database.
for trial_method in ("scattered", "grouped", "mixed", "random"):
    a, _ = select_positions(240, Random(SEED), method=trial_method,
                            rate=(0.05, 0.05), n_groups=3, grouped_share=0.70)
    b, _ = select_positions(240, Random(SEED), method=trial_method,
                            rate=(0.05, 0.05), n_groups=3, grouped_share=0.70)
    assert a == b and len(a) == 12


In [ ]:
# Savepoint check: a failed combined corruption must roll back fully.
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    source = sql.Identifier(PRACTICE_SCHEMA, PRACTICE_TABLE)
    target_id = sql.Identifier(ERROR_TARGET_COLUMN)
    before = conn.execute(sql.SQL("SELECT SUM({target}) FROM {source}").format(target=target_id, source=source)).fetchone()[0]
    try:
        with conn.transaction():
            rng = Random(SEED)
            changed, kind, placement = add_value_errors_sql(
                conn, rng, schema=PRACTICE_SCHEMA, table=PRACTICE_TABLE,
                time_column=TIME_COLUMN, key_column=KEY_COLUMN, timezone=TIMEZONE,
                start_date=START_DATE, end_date=END_DATE,
                target_column=ERROR_TARGET_COLUMN, kind=ERROR_KIND, placement=ERROR_PLACEMENT,
                error_rate=ERROR_RATE, n_groups=ERROR_GROUPS,
                grouped_share=ERROR_GROUPED_SHARE, bounds_share=ERROR_BOUNDS_SHARE,
                valid_min=VALID_MIN, valid_max=VALID_MAX,
                outside_offset=OUTSIDE_OFFSET, scale_factor=SCALE_FACTOR,
            )
            raise RuntimeError("Rollback test")
    except RuntimeError as exc:
        if str(exc) != "Rollback test":
            raise
    after = conn.execute(sql.SQL("SELECT SUM({target}) FROM {source}").format(target=target_id, source=source)).fetchone()[0]
    assert after == before
print("Rollback verified")

In [ ]:
# Apply and verify within one practice-table transaction.
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    rng = Random(SEED)
    changed, kind, placement = add_value_errors_sql(
        conn, rng, schema=PRACTICE_SCHEMA, table=PRACTICE_TABLE,
        time_column=TIME_COLUMN, key_column=KEY_COLUMN, timezone=TIMEZONE,
        start_date=START_DATE, end_date=END_DATE,
        target_column=ERROR_TARGET_COLUMN, kind=ERROR_KIND, placement=ERROR_PLACEMENT,
        error_rate=ERROR_RATE, n_groups=ERROR_GROUPS,
        grouped_share=ERROR_GROUPED_SHARE, bounds_share=ERROR_BOUNDS_SHARE,
        valid_min=VALID_MIN, valid_max=VALID_MAX,
        outside_offset=OUTSIDE_OFFSET, scale_factor=SCALE_FACTOR,
    )
    print("Value changes:", changed, "| kind:", kind, "| placement:", placement)

In [ ]:
# Inspect range violations. Scaling may be numerically plausible and require upstream evidence.
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    table_id = sql.Identifier(PRACTICE_SCHEMA, PRACTICE_TABLE)
    target_id = sql.Identifier(ERROR_TARGET_COLUMN)
    time_id = sql.Identifier(TIME_COLUMN)
    observations = conn.execute(sql.SQL("""
        SELECT {time}, {target} FROM {table}
        WHERE {target} < %s OR {target} > %s
        ORDER BY {time} LIMIT 20
    """).format(time=time_id, target=target_id, table=table_id), (VALID_MIN, VALID_MAX)).fetchall()
observations

In [ ]:
# A range violation is demonstrable if its domain rule is explicit.
assert changed > 0
# Numerically plausible scaled errors may require comparison with an authoritative source.

In [ ]:
# Developer diagnostic: documented range and observed out-of-range values.
import matplotlib.pyplot as plt
if observations:
    vals = pd.DataFrame(observations, columns=[TIME_COLUMN, ERROR_TARGET_COLUMN])
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.scatter(vals[TIME_COLUMN], vals[ERROR_TARGET_COLUMN], s=20)
    ax.axhline(VALID_MIN, linestyle="--", linewidth=1)
    ax.axhline(VALID_MAX, linestyle="--", linewidth=1)
    ax.set_title("Observed contract violations (sample)")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()